In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier

# ========================
# PERFORMANCE METRICS
# ========================
def performance_measures(conf_mat):

    accuracy = np.trace(conf_mat.values) / conf_mat.values.sum()

    tp = conf_mat.loc["Yes", "Yes"]
    fp = conf_mat.loc["Yes", "No"]
    tn = conf_mat.loc["No", "No"]
    fn = conf_mat.loc["No", "Yes"]

    sensitivity = np.nan if (tp + fn) == 0 else tp / (tp + fn)
    specificity = np.nan if (tn + fp) == 0 else tn / (tn + fp)
    precision = np.nan if (tp + fp) == 0 else tp / (tp + fp)

    if pd.isna(precision) or pd.isna(sensitivity) or (precision + sensitivity) == 0:
        f1_score = np.nan
    else:
        f1_score = 2 * precision * sensitivity / (precision + sensitivity)

    return {
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "F1_score": f1_score
    }


def get_metrics(predicted, actual):
    labels = ["No", "Yes"]
    cm = pd.DataFrame(
        confusion_matrix(actual, predicted, labels=labels),
        index=labels,
        columns=labels
    )
    cm.index.name = "Predicted"
    cm.columns.name = "Actual"
    return performance_measures(cm)

def min_max_scale(series, min_val, max_val):
    if max_val == min_val:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - min_val) / (max_val - min_val)

pneumonia_data=pd.read_csv(r"C:/Users/000110888/OneDrive - CSULB/Desktop/pneumonia_data.csv")

pneumonia_data["pneumonia"]=np.where(pneumonia_data["pneumonia"]=="yes", "Yes", "No")
pneumonia_data["pneumonia"]=pd.Categorical(pneumonia_data["pneumonia"], categories=["No", "Yes"])

pneumonia_data["gender"]=np.where(pneumonia_data["gender"]=="M", 1, 0)
pneumonia_data["tobacco_use"]=np.where(pneumonia_data["tobacco_use"]=="yes", 1, 0)

# ================================
# SPLITTING INTO TRAINING/TESTING
# ================================
train, test = train_test_split(pneumonia_data, test_size=0.2, 
stratify=pneumonia_data["pneumonia"], random_state=447033)

train=train.reset_index(drop=True)
test=test.reset_index(drop=True)

# =============================================================
# MIN-MAX SCALING OF ALL PREDICTORS USING TRAINING SET ONLY
# =============================================================
age_min=train["age"].min()
age_max=train["age"].max()
train["age"]=min_max_scale(train["age"], age_min, age_max)
test["age"]=min_max_scale(test["age"], age_min, age_max)

pm_min=train["PM2_5"].min()
pm_max=train["PM2_5"].max()
train["PM2_5"]=min_max_scale(train["PM2_5"], pm_min, pm_max)
test["PM2_5"]=min_max_scale(test["PM2_5"], pm_min, pm_max)

# preparing target variable for different models
train_y_factor=train["pneumonia"]
test_y_factor=test["pneumonia"]

# numeric outcome for xgboost, neural net, and meta-model
train_y_num=np.where(train["pneumonia"]=="Yes", 1, 0)
test_y_num=np.where(test["pneumonia"]=="Yes", 1, 0)

train_x=train.drop(columns=["pneumonia"]).to_numpy()
test_x=test.drop(columns=["pneumonia"]).to_numpy()

# ===================================
# MAKING OUT-OF-FOLD PREDICTIONS
# ===================================
skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=559702)

oof_rf=np.full(len(train), np.nan)
oof_xgb=np.full(len(train), np.nan)
oof_svm_linear=np.full(len(train), np.nan)
oof_svm_radial=np.full(len(train), np.nan)
oof_knn=np.full(len(train), np.nan)
oof_nb=np.full(len(train), np.nan)
oof_ann=np.full(len(train), np.nan)

for i, (train_idx, valid_idx) in enumerate(skf.split(train.drop(columns=["pneumonia"]), train_y_factor), start=1):
    print(f"\nProcessing fold {i} of 5 ...")

    fold_train=train.iloc[train_idx].reset_index(drop=True)
    fold_valid=train.iloc[valid_idx].reset_index(drop=True)

    fold_train_x=fold_train.drop(columns=["pneumonia"]).to_numpy()
    fold_valid_x=fold_valid.drop(columns=["pneumonia"]).to_numpy()

    fold_train_y_factor=fold_train["pneumonia"]
    fold_valid_y_factor=fold_valid["pneumonia"]

    fold_train_y_num=np.where(fold_train["pneumonia"]=="Yes", 1, 0)

    # random forest
    model_rf=RandomForestClassifier(n_estimators=150,  
        max_features=min(4, fold_train.shape[1]-1),
        max_leaf_nodes=30, random_state=559702)
    
    model_rf.fit(fold_train.drop(columns=["pneumonia"]), fold_train_y_factor)
    oof_rf[valid_idx]=model_rf.predict_proba(fold_valid.drop(columns=["pneumonia"]))[:,1]

    # xgboost
    model_xgb=XGBClassifier(objective="binary:logistic", eval_metric="auc",
    max_depth=6, learning_rate=0.01, subsample=0.8, colsample_bytree=0.5,
    n_estimators=300, random_state=559702, verbosity=0)
    
    model_xgb.fit(fold_train_x, fold_train_y_num)
    oof_xgb[valid_idx]=model_xgb.predict_proba(fold_valid_x)[:,1]

    # svm linear
    model_svm_linear=SVC(kernel="linear", probability=True, random_state=559702)
    model_svm_linear.fit(fold_train_x, fold_train_y_factor)
    oof_svm_linear[valid_idx]=model_svm_linear.predict_proba(fold_valid_x)[:,1]

    # svm radial
    model_svm_radial=SVC(kernel="rbf", probability=True, random_state=559702)
    model_svm_radial.fit(fold_train_x, fold_train_y_factor)
    oof_svm_radial[valid_idx]=model_svm_radial.predict_proba(fold_valid_x)[:,1]

    # knn
    knn_grid=GridSearchCV(estimator=KNeighborsClassifier(),
        param_grid={"n_neighbors": list(range(5, 16))},
        cv=5, scoring="roc_auc")
    
    knn_grid.fit(fold_train.drop(columns=["pneumonia"]), fold_train_y_factor)
    model_knn=knn_grid.best_estimator_
    oof_knn[valid_idx]=model_knn.predict_proba(fold_valid.drop(columns=["pneumonia"]))[:,1]

    # naive bayes
    model_nb=GaussianNB()
    model_nb.fit(fold_train_x, fold_train_y_factor)
    oof_nb[valid_idx]=model_nb.predict_proba(fold_valid_x)[:,1]

    # artificial neural network
    model_ann = MLPClassifier(hidden_layer_sizes=(3,), activation="logistic",
    max_iter=2000, random_state=559702)

    try:
        model_ann.fit(fold_train_x, fold_train_y_num)
        ann_prob=model_ann.predict_proba(fold_valid_x)[:, 1]
    except Exception:
        ann_prob=np.repeat(np.mean(fold_train_y_num), len(fold_valid))

    oof_ann[valid_idx]=ann_prob

# ========================
# FITTING META-MODEL
# ========================
stack_train=pd.DataFrame({
    "rf": oof_rf,
    "xgb": oof_xgb,
    "svm_linear": oof_svm_linear,
    "svm_radial": oof_svm_radial,
    "knn": oof_knn,
    "nb": oof_nb,
    "ann": oof_ann,
    "pneumonia": train_y_num
})

meta_model=LogisticRegression(max_iter=2000)
meta_model.fit(stack_train.drop(columns=["pneumonia"]), stack_train["pneumonia"])

# ==============================
# FITTING MODELS ON FULL TRAIN
# ==============================
rf_biclass=RandomForestClassifier(n_estimators=150, max_features=min(4, train.shape[1]-1),
max_leaf_nodes=30, random_state=447033)
rf_biclass.fit(train.drop(columns=["pneumonia"]), train_y_factor)

xgb_biclass=XGBClassifier(objective="binary:logistic", eval_metric="auc",
max_depth=6, learning_rate=0.01, subsample=0.8, colsample_bytree=0.5,
n_estimators=300, random_state=447033, verbosity=0)
xgb_biclass.fit(train_x, train_y_num)

svm_class_linear=SVC(kernel="linear", probability=True, random_state=447033)
svm_class_linear.fit(train_x, train_y_factor)

svm_class_radial=SVC(kernel="rbf", probability=True, random_state=447033)
svm_class_radial.fit(train_x, train_y_factor)

knn_grid_full=GridSearchCV(estimator=KNeighborsClassifier(),
    param_grid={"n_neighbors": list(range(5, 16))},
    cv=5, scoring="roc_auc")

knn_grid_full.fit(train.drop(columns=["pneumonia"]), train_y_factor)
knn_biclass=knn_grid_full.best_estimator_

nb_biclass=GaussianNB()
nb_biclass.fit(train_x, train_y_factor)

ann_biclass=MLPClassifier(hidden_layer_sizes=(3,), activation="logistic",
max_iter=2000, random_state=447033)

ann_biclass.fit(train_x, train_y_num)

# ========================
# PREDICTING ON TEST SET
# ========================
test_rf=rf_biclass.predict_proba(test.drop(columns=["pneumonia"]))[:, 1]
test_xgb=xgb_biclass.predict_proba(test_x)[:, 1]
test_svm_linear=svm_class_linear.predict_proba(test_x)[:, 1]
test_svm_radial=svm_class_radial.predict_proba(test_x)[:, 1]
test_knn=knn_biclass.predict_proba(test.drop(columns=["pneumonia"]))[:, 1]
test_nb=nb_biclass.predict_proba(test_x)[:, 1]

try:
    test_ann=ann_biclass.predict_proba(test_x)[:, 1]
except Exception:
    test_ann=np.repeat(np.mean(train_y_num), len(test))

# ========================
# STACKING PREDICTIONS
# ========================
stack_test=pd.DataFrame({
    "rf": test_rf,
    "xgb": test_xgb,
    "svm_linear": test_svm_linear,
    "svm_radial": test_svm_radial,
    "knn": test_knn,
    "nb": test_nb,
    "ann": test_ann
})

stack_prob=meta_model.predict_proba(stack_test)[:,1]
stack_class=np.where(stack_prob >= 0.5, "Yes", "No")
stack_class=pd.Categorical(stack_class, categories=["No", "Yes"])

# confusion matrix
conf_mat=pd.DataFrame(confusion_matrix(test_y_factor, stack_class, labels=["No", "Yes"]),
index=["No", "Yes"], columns=["No", "Yes"])
conf_mat.index.name = "Predicted"
conf_mat.columns.name = "Actual"

# performance measures
m=performance_measures(conf_mat)

# ========================
# INDIVIDUAL MODEL METRICS
# ========================
pred_class_rf=pd.Categorical(np.where(test_rf>=0.5, "Yes", "No"), categories=["No", "Yes"])
pred_class_xgb=pd.Categorical(np.where(test_xgb>=0.5, "Yes", "No"), categories=["No", "Yes"])
pred_class_svm_linear=pd.Categorical(np.where(test_svm_linear>=0.5, "Yes", "No"), categories=["No", "Yes"])
pred_class_svm_radial=pd.Categorical(np.where(test_svm_radial>=0.5, "Yes", "No"), categories=["No", "Yes"])
pred_class_knn=pd.Categorical(np.where(test_knn>=0.5, "Yes", "No"), categories=["No", "Yes"])
pred_class_nb=pd.Categorical(np.where(test_nb>=0.5, "Yes", "No"), categories=["No", "Yes"])
pred_class_ann=pd.Categorical(np.where(test_ann>=0.5, "Yes", "No"), categories=["No", "Yes"])

m_rf=get_metrics(pred_class_rf, test_y_factor)
m_xgb=get_metrics(pred_class_xgb, test_y_factor)
m_svm_linear=get_metrics(pred_class_svm_linear, test_y_factor)
m_svm_radial=get_metrics(pred_class_svm_radial, test_y_factor)
m_knn=get_metrics(pred_class_knn, test_y_factor)
m_nb=get_metrics(pred_class_nb, test_y_factor)
m_ann=get_metrics(pred_class_ann, test_y_factor)
m_stack=get_metrics(stack_class, test_y_factor)

results=pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "SVM Linear",
        "SVM Radial",
        "KNN",
        "Naive Bayes",
        "ANN",
        "Stacked"
    ],
    "Accuracy": [
        m_rf["accuracy"], m_xgb["accuracy"], m_svm_linear["accuracy"],
        m_svm_radial["accuracy"], m_knn["accuracy"], m_nb["accuracy"],
        m_ann["accuracy"], m_stack["accuracy"]
    ],
    "Sensitivity": [
        m_rf["sensitivity"], m_xgb["sensitivity"], m_svm_linear["sensitivity"],
        m_svm_radial["sensitivity"], m_knn["sensitivity"], m_nb["sensitivity"],
        m_ann["sensitivity"], m_stack["sensitivity"]
    ],
    "Specificity": [
        m_rf["specificity"], m_xgb["specificity"], m_svm_linear["specificity"],
        m_svm_radial["specificity"], m_knn["specificity"], m_nb["specificity"],
        m_ann["specificity"], m_stack["specificity"]
    ],
    "Precision": [
        m_rf["precision"], m_xgb["precision"], m_svm_linear["precision"],
        m_svm_radial["precision"], m_knn["precision"], m_nb["precision"],
        m_ann["precision"], m_stack["precision"]
    ],
    "F1_score": [
        m_rf["F1_score"], m_xgb["F1_score"], m_svm_linear["F1_score"],
        m_svm_radial["F1_score"], m_knn["F1_score"], m_nb["F1_score"],
        m_ann["F1_score"], m_stack["F1_score"]
    ]
})

results.iloc[:, 1:]=results.iloc[:, 1:].round(4)

print("Performance comparison table:")
print(results)